# M02A: Zero-Shot and One-Shot Prompting

You can make API calls. Now let's make them effective.

**What you'll learn:**
- The Instruction Formula (role, task, constraints, format)
- Zero-shot prompting (no examples needed)
- One-shot prompting (single example guides output)

**What you'll build:**
- Text classifiers using both techniques
- Side-by-side comparisons

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


def ask_openai(prompt, model=MODEL):
    """Ask OpenAI a question using the global client."""
    
    try:
        response = client.responses.create(
            model=model,
            input=prompt
        )
        return response.output_text.strip()
        
    except openai.AuthenticationError:
        return "Error: Invalid API key. Check your .env file."
    except openai.RateLimitError:
        return "Error: Rate limit exceeded. Wait and try again."
    except openai.APIConnectionError:
        return "Error: Network issue. Check your internet."
    except openai.BadRequestError:
        return "Error: Bad request. Check model name."
    except Exception as e:
        return f"API Error: {str(e)}"


print("✅ Ready!")

---

## 🎯 Part 1: Zero-Shot Prompting

**Zero-shot prompting** means asking the AI to perform a task without providing any examples.

### How it works:
```
Prompt → Model (no examples) → Output
```

### When to use zero-shot
- Simple, well-known tasks (translation, summarization)
- When you don't have examples
- Quick prototyping

Let's see zero-shot in action with a text classification task.

### 🔬 Example 1: The "Lazy" Zero-Shot

Let's try a very basic prompt without specific instructions:

In [ ]:
# Zero-shot prompt - Vague instructions
lazy_prompt = """What is the sentiment of this customer feedback?

Customer feedback: "The product works as expected."""

print("=== Lazy Zero-Shot ===\n")
print("Prompt:")
print(lazy_prompt)
print("\n" + "="*50 + "\n")

print("AI Response:")
print(ask_openai(lazy_prompt))

### ⚠️ Notice the Issues

Without clear guidance, the AI often gives:
- **Explanations instead of labels** - "The sentiment is positive because..."
- **Inconsistent terminology** - good/positive/favorable/satisfied
- **Uncertainty** - "This seems to be..."

**This breaks your code:**
```python
if sentiment == "positive":  # ❌ Fails - strings don't match
    return "😊"
```

If the AI returns `"The sentiment is positive"` instead of `"positive"`, the comparison fails and your logic breaks.

---

## ✍️ Part 2: The Instruction Formula

Before we add examples (One-Shot), we can fix many problems just by writing better instructions.

A good instruction has four parts:

1. **Role** (optional): Who should the model act as? (e.g., "You are a data analyst")
2. **Task**: What exactly should it do? (e.g., "Classify sentiment")
3. **Constraints**: What should it avoid or limit? (e.g., "Do not explain")
4. **Format**: How should the output look? (e.g., "One single word")

Let's try applying this formula to our Zero-Shot prompt.

In [ ]:
# Improved Zero-Shot Prompt using the Formula
improved_prompt = """You are a sentiment analysis system. (Role)
Classify the sentiment of this customer feedback. (Task)
Respond with exactly one word: positive, negative, or neutral. (Format)
Do not provide explanations. (Constraint)

Customer feedback: "The product works as expected."""

print("=== Improved Zero-Shot (With Formula) ===\n")
print(ask_openai(improved_prompt))

### ✅ Much Better!

By adding **Constraints** and **Format**, we got a clean single-word label.

However, even with good instructions, Zero-shot can still struggle with complex formats or specific styles. That is where **One-Shot Prompting** comes in.

---

## 🎯 Part 3: One-Shot Prompting

**One-shot prompting** means providing ONE example of the input-output format you want.

### The pattern:
```
Task description

Example:
Input: [example input]
Output: [example output]

Now classify this:
Input: [your actual input]
```

This "shows" the AI what you want, rather than just "telling" it.

### 🔬 Example 3: One-Shot Text Classification

Let's use an example to force the AI to follow our format exactly.

In [ ]:
# One-shot prompt - Includes ONE example
one_shot_prompt = """What is the sentiment of this customer feedback.

Example:
Feedback: "Great product, highly recommend!"
positive

Now classify this:
Feedback: "The product works as expected."""

print("=== One-Shot Classification ===\n")
print("Prompt:")
print(one_shot_prompt)
print("\n" + "="*50 + "\n")

print("AI Response:")
print(ask_openai(one_shot_prompt))

### 🔬 Example 4: Testing Multiple Cases

Let's test if our One-Shot prompt holds up against different types of feedback.

In [ ]:
# Test one-shot on multiple examples
feedback_cases = [
    "The product works as expected.",
    "Amazing quality, exceeded my expectations!",
    "Broke after two days of use.",
    "It's fine.",
    "Fast shipping but average product."
]

print("=== One-Shot: Multiple Feedback Cases ===\n")

for i, feedback in enumerate(feedback_cases, 1):
    prompt = f"""Classify the sentiment of this customer feedback.

Example:
Feedback: "Great product, highly recommend!"
positive

Now classify this:
Feedback: "{feedback}"
"""
    
    print(f"{i}. Feedback: {feedback}")
    print(f"   Sentiment: ", end="")
    print(ask_openai(prompt))
    print("=" * 60 + "\n")

### 📊 Observations: One-Shot Performance

Notice the improvements:
- ✅ **Consistent format** - No extra punctuation or text.
- ✅ **Implicit Instructions** - We didn't have to say "Don't explain." The example *showed* it shouldn't explain.

**One example makes responses significantly more consistent and accurate!**

---

## 📊 Side-by-Side Comparison

Let's directly compare the three methods we've learned:

In [ ]:
example_feedback = "Fast shipping but average product."

print(f"Example feedback: '{example_feedback}'\n")
print("="*60)

# 1. Lazy Zero-Shot
print("\n🔴 1. Lazy Zero-Shot:")
prompt1 = f"What is the sentiment of: '{example_feedback}'"
print(ask_openai(prompt1))

# 2. Improved Zero-Shot (Instruction Formula)
print("\n🟡 2. Improved Zero-Shot (Formula):")
prompt2 = f"Classify sentiment (positive/negative/neutral). One word only. Input: '{example_feedback}'"
print(ask_openai(prompt2))

# 3. One-Shot
print("\n🟢 3. One-Shot (Example):")
prompt3 = f"""Classify sentiment.
Input: "Great product!"
positive

Input: "{example_feedback}"""
print(ask_openai(prompt3))

print("\n" + "="*60)

---

## 💡 Practical Tips

### When to use Zero-Shot
✅ Simple, common tasks (translation, summarization)  
✅ Quick prototyping  
✅ General knowledge questions  

### When to use One-Shot
✅ You need consistent formatting  
✅ Specialized or domain-specific tasks  
✅ Building production systems  

### Crafting Good Examples
Your example should show:
- **Input format** — How the data looks
- **Output format** — Exact format you want (length, style, labels)
- **Style** — Tone, capitalization, punctuation

---

## 💪 Practice Exercises

Now it's your turn! Try these exercises to practice zero-shot and one-shot prompting.

### Exercise 1: Email Subject Line Generation (Solution Provided)

Create a one-shot prompt to generate concise email subject lines from email content.

**Your example should show:**
- Email content as input
- Short, descriptive subject line as output

**Here's a working solution to learn from:**

In [ ]:
# Exercise 1: Email Subject Line Generation

email_content = "Hi team, I wanted to share the Q4 sales results. We exceeded our target by 15% and saw strong growth in the enterprise segment. Great job everyone!"

prompt = f"""Generate a concise email subject line.

Example:
Email: "Hi Sarah, I've attached the updated project timeline. The launch date has moved to March 15th due to the new feature requests. Let me know if you have questions."
Project Timeline Updated - Launch Moved to March 15

Now generate a subject for this email:
Email: "{email_content}"
"""

print("Email:")
print(email_content)
print("\n" + "="*60 + "\n")

print("Generated Subject: ", end="")
print(ask_openai(prompt))

### Exercise 2: Intent Detection

Create a one-shot prompt to classify user intent into categories such as: question, request, complaint, or feedback.

**Your example should show:**
- User message as input
- Intent category as output (one word)

In [ ]:
# Exercise 2: Intent Detection
# Objective: Classify user messages into: Question, Request, Complaint, or Feedback.

user_messages = [
    "How do I reset my password?",
    "I'd like to cancel my subscription",
    "The app keeps crashing on my phone",
    "Love the new features!"
]

print("=== Exercise 2: Intent Detection ===\n")

# TODO 1: Loop through the 'user_messages' list
# (Hint: for msg in user_messages:)
    
    # TODO 2: Create your One-Shot prompt.
    # Make sure to include ONE example (e.g., "Where is my invoice?" -> Question)
    # Then ask it to classify the current 'msg' variable.
    
    # TODO 3: Call ask_openai() with your prompt
    
    # TODO 4: Print the message and the result
    
# --- Write your code below this line ---

### Exercise 3: Code Comment Generator

Create a one-shot prompt to generate concise inline comments for code.

**Your example should show:**
- Code snippet as input
- Concise comment as output

In [ ]:
# Exercise 3: Code Comment Generator
# Objective: Create a One-Shot prompt that generates a comment for a specific line of code.

code_snippet = "result = [x**2 for x in range(10) if x % 2 == 0]"

print("=== Exercise 3: Code Commenter ===\n")
print(f"Input Code:\n{code_snippet}\n")

# TODO 1: Create your One-Shot prompt.
# Give an example of code and its comment (e.g., "x=x+1" -> "# Increment x")
# Then ask for a comment for the 'code_snippet' variable defined above.

# TODO 2: Call ask_openai() with your prompt

# TODO 3: Print the generated comment

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

### What You Learned

**The Instruction Formula:**
- **Role + Task + Constraints + Format**
- Use this to improve Zero-shot performance.

**Zero-Shot Prompting:**
- Works well for general tasks, but can be verbose.
- Use it when you don't have examples.

**One-Shot Prompting:**
- Provide ONE example of desired input/output.
- Noticeably better consistency and adherence to format.
- **Key Insight:** Examples "show" the AI what to do better than instructions can "tell" it.

---

### 📍 Next Step

**M02B: Few-Shot Prompting** — Boost accuracy with examples:
- Build effective example sets
- Find the sweet spot (3-5 examples)
- Handle edge cases

---

## 🔧 Troubleshooting

**API not responding?**
- Run Step 1: Setup at the top
- Verify .env file has your API key

**Getting inconsistent results?**
- This is normal for zero-shot! Try adding "One Word Only" to your prompt.
- If that fails, move to One-Shot prompting.

**Responses too verbose?**
- Check your constraints (e.g., "Do not explain").
- Check your example (is your example response verbose?)

---